# Calgary 311 и историски временски податоци

Оваа тетратка е водич низ целосната анализа. Таа не го сведува проектот на избор на алгоритам, туку ги користи моделите за да одговори на четири истражувачки прашања: (1) како времето е поврзано со дневниот број snow/ice барања, (2) дали може да се препознаат денови со невообичаено висока побарувачка, (3) дали при поднесување може да се предвиди затворање во рок од 7 дена и (4) дали постојат интерпретабилни профили на заедници.

**Извори:** City of Calgary Open Data и Open-Meteo Historical Weather API. Деталните методолошки одлуки, литературата и ограничувањата се во `README.md` и во семинарската работа.

## 1. Репродукција

Стандардно тетратката ги чита веќе создадените резултати. За целосно повторување поставете `REBUILD = True`. Преземањето бара интернет и може да трае; потоа анализата ги создава обработените табели, моделите, метриките и графиците.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import display, Markdown, SVG

ROOT = Path.cwd()
if not (ROOT / 'README.md').exists():
    raise RuntimeError('Стартувајте ја тетратката од коренот на проектот.')
REBUILD = False

if REBUILD:
    subprocess.run([sys.executable, str(ROOT / 'work' / 'download_project_data.py')], check=True)
    subprocess.run([sys.executable, str(ROOT / 'work' / 'analyze_project.py')], check=True)

RESULTS_PATH = ROOT / 'outputs' / 'analysis_results.json'
METRICS_PATH = ROOT / 'outputs' / 'model_metrics.csv'
assert RESULTS_PATH.exists() and METRICS_PATH.exists(), 'Поставете REBUILD=True или обезбедете ги готовите outputs.'

## 2. Податоци, променливи и заштита од leakage

Snow/ice барањата се агрегираат по локален календарски ден и се поврзуваат со дневното време за Калгари (51.0447, −114.0719; `America/Edmonton`). За Q1 целта е број на барања; за Q2 бинарна ознака над 95-тиот перцентил пресметан **само од тренинг-периодот**; за Q3 целта е дали времето до затворање е најмногу 7 дена; за Q4 заедниците се претставени со нормализиран состав на типови барања и мерки за интензитет/сезоналност.

Временски поделби: Q1/Q2 — train 2018–2023, validation 2024, test 2025; Q3 — train 2021–2023, validation 2024, test 2025. Препроцесирањето, ретките категории, праговите и хиперпараметрите се определуваат без test-податоците. Датумот на затворање се користи само за создавање на целта, никогаш како predictor.

In [ ]:
with RESULTS_PATH.open(encoding='utf-8') as f:
    results = json.load(f)
metrics = pd.read_csv(METRICS_PATH)

display(pd.Series(results['data'], name='вредност').to_frame())
display(metrics)

## 3. Експлораторна анализа

Дневните барања имаат силна сезоналност, многу мирни денови и кратки екстремни врвови. Затоа само случајна поделба би дала премногу оптимистична слика. Месечниот приказ подолу ја покажува заедничката зимска структура, но не докажува причинско-последична врска.

In [ ]:
display(SVG(filename=str(ROOT / 'outputs' / 'figures' / 'monthly_requests_and_snowfall.svg')))

## 4. Q1 — дневен број snow/ice барања

Се споредуваат сезонски baseline, Poisson регресија и Random Forest. MAE е главна практична мерка (просечна апсолутна грешка во број барања), R² ја мери објаснетата варијација, а Poisson deviance е соодветна за count-цел. Poisson моделот има test MAE 67.7 наспроти 105.2 за baseline и R² 0.296. Резултатот поддржува умерена предвидливост: сезоната и задоцнетата температура се најсилни, а тековната температура и снегот додаваат сигнал. Екстремните врвови остануваат тешки за предвидување.

In [ ]:
display(metrics.loc[metrics['question'].eq('Q1 count'), ['model', 'mae', 'rmse', 'r2', 'poisson_deviance']])
display(SVG(filename=str(ROOT / 'outputs' / 'figures' / 'snow_regression_importance.svg')))

## 5. Q2 — денови со висока побарувачка

„Висока побарувачка“ е дефинирана како најмалку 480 барања дневно, односно 95-тиот перцентил на тренинг-периодот. Во 2025 такви се само 2.36% од деновите, па PR-AUC, precision, recall, F1 и balanced accuracy се поважни од обична accuracy. Weather-only Random Forest има ROC-AUC 0.890 и PR-AUC 0.142; фаќа 80% од високите денови, но precision е 9.3%. Значи времето е употребливо за широка рана тревога, но не и за тесно оперативно алармирање без многу лажни тревоги.

In [ ]:
display(metrics.loc[metrics['question'].eq('Q2 high demand'), ['model', 'roc_auc', 'pr_auc', 'precision', 'recall', 'f1', 'balanced_accuracy']])
display(SVG(filename=str(ROOT / 'outputs' / 'figures' / 'high_demand_importance.svg')))

## 6. Q3 — затворање во рок од 7 дена

Предвидувањето користи само service name, agency, source и календарски информации достапни при поднесување. Логистичката регресија има ROC-AUC 0.925 и PR-AUC 0.299 и дава попрецизно рангирање (precision 0.446, recall 0.355). Дрвото жртвува precision за многу повисок recall 0.773. Најважни се видот на услугата и надлежната служба; каналот и календарот се помали сигнали. Ова е модел на административно „Closed“, не гаранција дека физичката работа е завршена.

In [ ]:
display(metrics.loc[metrics['question'].eq('Q3 slow closure'), ['model', 'roc_auc', 'pr_auc', 'precision', 'recall', 'f1', 'balanced_accuracy']])
q3 = results.get('closure', {})
display(q3)

## 7. Q4 — профили на заедници

K-means со k=3 создава три разбирливи профили: мал број нови/растечки заедници со варијабилна побарувачка; голема група воспоставени и повисоко-обемни заедници со waste/cart барања; и урбани/мобилност ориентирани заедници. Silhouette 0.282 покажува дека границите не се остри. Стабилноста со Ward clustering е добра (ARI 0.732), но временската стабилност е умерена (~0.41). Бидејќи нема сигурен населeнски denominator по година, „интензитет“ не треба да се толкува како стапка по жител.

In [ ]:
q4 = results.get('clustering', {})
display(q4)

## 8. Заклучок, ограничувања и употреба на LLM

Времето реално објаснува дел од snow/ice побарувачката, но однесувањето на граѓаните, оперативните политики и неевидентирани локални услови остануваат важни. Ретки high-demand денови може да се детектираат со висок recall, но со многу false positives. Брзината на затворање најмногу зависи од service/agency. Кластерите се корисни описни профили, но не се природни и непроменливи категории.

Главни ограничувања: reanalysis време за една градска точка; набљудувачка, не причинска анализа; можни промени во 311 политиките; административно значење на `Closed`; само пет high-demand test денови; нема population denominator и географска форма за кластерите.

LLM беше користен како помош за планирање, проверка на методологијата, пишување документација и организација на кодот. Не беше извор на резултати: бројките се создадени од скриптите, а одлуките се проверливи во `work/`, `outputs/` и наведената литература. Одговорноста за толкувањето и финалната проверка останува кај авторот.